# XCCY Swap Pricing - Análisis Detallado

Este notebook analiza paso a paso cómo funciona `XccySwapPricer`:

1. **Inputs** que recibe la función `price()`
2. **Construcción del schedule** de pagos
3. **Amortización** (bullet vs linear vs custom)
4. **Valoración de cada pata** (USD SOFR / COP IBR)
5. **Notional exchange** PV
6. **NPV final** y conversión FX
7. **Mid-life pricing** (swaps ya iniciados)
8. **P&L decomposition** y sensibilidades

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from dotenv import load_dotenv
load_dotenv(os.path.join('..', '.env'), override=True)

import QuantLib as ql
import pandas as pd
from datetime import datetime

from pricing.data.market_data import MarketDataLoader
from pricing.curves.curve_manager import CurveManager
from pricing.instruments.xccy_swap import (
    XccySwapPricer,
    build_amortization_schedule,
    validate_amortization_schedule,
)
from utilities.date_functions import datetime_to_ql, ql_to_datetime

pd.set_option('display.float_format', '{:.6f}'.format)

# Construir curvas
loader = MarketDataLoader()
cm = CurveManager()
cm.build_all(loader)
print(f'Valuation date: {cm.valuation_date}')
print(f'FX Spot: {cm.fx_spot}')
print(f'IBR curve: {"OK" if cm.ibr_curve else "MISSING"}')
print(f'SOFR curve: {"OK" if cm.sofr_curve else "MISSING"}')

---
## 1. Inputs de `XccySwapPricer.price()`

La función recibe estos parámetros:

| Parámetro | Tipo | Descripción |
|---|---|---|
| `notional_usd` | float | Nocional en USD |
| `start_date` | date/ql.Date | Fecha inicio del swap |
| `maturity_date` | date/ql.Date | Fecha vencimiento |
| `xccy_basis_bps` | float | Basis spread en bps (pata COP) |
| `pay_usd` | bool | True = pago USD, recibo COP |
| `fx_initial` | float | FX a inception (para notional exchange) |
| `cop_spread_bps` | float | Spread adicional pata COP |
| `usd_spread_bps` | float | Spread pata USD (ej: SOFR - 22bps) |
| `payment_frequency` | ql.Period | Frecuencia de pago (default 3M) |
| `amortization_type` | str | 'bullet', 'linear', 'custom' |
| `amortization_schedule` | list | Factores custom por período |

In [ ]:
# Ejemplo: CCS USD/COP bullet a 5 años
xccy = XccySwapPricer(cm)

# Parámetros del swap
params = {
    'notional_usd': 10_000_000,
    'start_date': datetime(2025, 3, 5),
    'maturity_date': datetime(2030, 3, 5),
    'xccy_basis_bps': 0.0,
    'pay_usd': True,
    'fx_initial': 4150.0,
    'usd_spread_bps': -22.0,   # SOFR - 22bps (típico Bancolombia)
    'cop_spread_bps': 0.0,
    'amortization_type': 'bullet',
}

print('=== Parámetros del Swap ===')
for k, v in params.items():
    print(f'  {k}: {v}')

---
## 2. Construcción del Schedule de Pagos

In [ ]:
# Replicar la construcción interna del schedule
start_ql = datetime_to_ql(params['start_date'])
mat_ql = datetime_to_ql(params['maturity_date'])

cop_cal = cm.ibr_index.fixingCalendar()
usd_cal = cm.sofr_index.fixingCalendar()
joint_cal = ql.JointCalendar(cop_cal, usd_cal)

schedule = ql.Schedule(
    start_ql, mat_ql,
    ql.Period(3, ql.Months),
    joint_cal,
    ql.Following, ql.Following,
    ql.DateGeneration.Forward, False,
)

dates = list(schedule)
print(f'Número de fechas en schedule: {len(dates)}')
print(f'Número de períodos: {len(dates) - 1}')
print()

schedule_df = pd.DataFrame([
    {
        'period': i,
        'start': ql_to_datetime(dates[i-1]).strftime('%Y-%m-%d'),
        'end': ql_to_datetime(dates[i]).strftime('%Y-%m-%d'),
        'days': ql.Actual360().dayCount(dates[i-1], dates[i]),
        'year_frac': ql.Actual360().yearFraction(dates[i-1], dates[i]),
    }
    for i in range(1, len(dates))
])
schedule_df

---
## 3. Amortización: Bullet vs Linear vs Custom

In [ ]:
notional_usd = params['notional_usd']
n_periods = len(dates) - 1

# Bullet
bullet = build_amortization_schedule(schedule, notional_usd, 'bullet')

# Linear
linear = build_amortization_schedule(schedule, notional_usd, 'linear')

# Custom ejemplo: mantiene 100% 8 períodos, luego baja 20% cada trimestre
custom_factors = [1.0]*8 + [0.8, 0.6, 0.4, 0.2] + [0.2]*(n_periods - 12)
custom_factors = custom_factors[:n_periods]
custom = build_amortization_schedule(schedule, notional_usd, 'custom', custom_factors)

amort_df = pd.DataFrame({
    'period': range(1, n_periods+1),
    'bullet': bullet,
    'linear': linear,
    'custom': custom,
})
amort_df['bullet_M'] = amort_df['bullet'] / 1e6
amort_df['linear_M'] = amort_df['linear'] / 1e6
amort_df['custom_M'] = amort_df['custom'] / 1e6
print(f'Períodos: {n_periods}')
amort_df[['period', 'bullet_M', 'linear_M', 'custom_M']]

In [ ]:
# Validar un schedule custom
validation = validate_amortization_schedule(custom_factors)
print('Validación del schedule custom:')
for k, v in validation.items():
    print(f'  {k}: {v}')

---
## 4. Valoración de cada pata (paso a paso)

Para cada período:
1. Forward rate desde la curva
2. + spread
3. Cashflow = notional * (fwd + spread) * tau
4. PV = cashflow * DF(end)

In [ ]:
# Pata USD (SOFR + spread)
usd_spread = params['usd_spread_bps'] / 10_000
cop_spread = (params['xccy_basis_bps'] + params['cop_spread_bps']) / 10_000
fx = params['fx_initial']
dc = ql.Actual360()

usd_notionals = build_amortization_schedule(schedule, notional_usd, 'bullet')
cop_notionals = [n * fx for n in usd_notionals]

usd_leg_rows = []
cop_leg_rows = []

for i in range(1, len(dates)):
    start = dates[i-1]
    end = dates[i]
    tau = dc.yearFraction(start, end)
    
    # USD
    fwd_usd = cm.sofr_handle.forwardRate(start, end, dc, ql.Simple).rate()
    cf_usd = usd_notionals[i-1] * (fwd_usd + usd_spread) * tau
    df_usd = cm.sofr_handle.discount(end)
    pv_usd = cf_usd * df_usd
    
    usd_leg_rows.append({
        'period': i,
        'start': ql_to_datetime(start).strftime('%Y-%m-%d'),
        'end': ql_to_datetime(end).strftime('%Y-%m-%d'),
        'notional': usd_notionals[i-1],
        'fwd_rate': fwd_usd * 100,
        'spread': usd_spread * 100,
        'all_in_rate': (fwd_usd + usd_spread) * 100,
        'tau': tau,
        'cashflow': cf_usd,
        'df': df_usd,
        'pv': pv_usd,
    })
    
    # COP
    fwd_cop = cm.ibr_handle.forwardRate(start, end, dc, ql.Simple).rate()
    cf_cop = cop_notionals[i-1] * (fwd_cop + cop_spread) * tau
    df_cop = cm.ibr_handle.discount(end)
    pv_cop = cf_cop * df_cop
    
    cop_leg_rows.append({
        'period': i,
        'start': ql_to_datetime(start).strftime('%Y-%m-%d'),
        'end': ql_to_datetime(end).strftime('%Y-%m-%d'),
        'notional': cop_notionals[i-1],
        'fwd_rate': fwd_cop * 100,
        'spread': cop_spread * 100,
        'all_in_rate': (fwd_cop + cop_spread) * 100,
        'tau': tau,
        'cashflow': cf_cop,
        'df': df_cop,
        'pv': pv_cop,
    })

usd_leg_df = pd.DataFrame(usd_leg_rows)
cop_leg_df = pd.DataFrame(cop_leg_rows)

print('=== PATA USD (SOFR + spread) ===')
print(f'USD spread: {usd_spread*10000:.0f} bps')
print(f'PV total pata USD: {usd_leg_df["pv"].sum():,.2f} USD')
usd_leg_df

In [ ]:
print('=== PATA COP (IBR + basis + spread) ===')
print(f'COP spread total: {cop_spread*10000:.0f} bps')
print(f'PV total pata COP: {cop_leg_df["pv"].sum():,.2f} COP')
cop_leg_df

---
## 5. Notional Exchange PV

Para un swap bullet:
- **Inicio**: Pago USD notional / Recibo COP notional
- **Final**: Recibo USD notional / Pago COP notional

Para un swap amortizable se agregan retornos intermedios.

In [ ]:
# Notional exchange PV (bullet)
notional_cop = notional_usd * fx

# Desde la perspectiva del payer USD:
# Inicio: paga USD (negativo), recibe COP (positivo) — ya liquidado si mid-life
# Final: recibe USD (positivo), paga COP (negativo)

df_start_usd = cm.sofr_handle.discount(dates[0])
df_end_usd = cm.sofr_handle.discount(dates[-1])
df_start_cop = cm.ibr_handle.discount(dates[0])
df_end_cop = cm.ibr_handle.discount(dates[-1])

print('=== Discount Factors para Notional Exchange ===')
print(f'Start ({ql_to_datetime(dates[0]).strftime("%Y-%m-%d")}):')
print(f'  SOFR DF: {df_start_usd:.8f}')
print(f'  IBR  DF: {df_start_cop:.8f}')
print(f'End ({ql_to_datetime(dates[-1]).strftime("%Y-%m-%d")}):')
print(f'  SOFR DF: {df_end_usd:.8f}')
print(f'  IBR  DF: {df_end_cop:.8f}')
print()

# La función internamente usa _notional_exchange_pv_amort
# Para bullet: PV = -N*DF(start) + N*DF(end)
usd_nex_pv_raw = -notional_usd * df_start_usd + notional_usd * df_end_usd
cop_nex_pv_raw = -notional_cop * df_start_cop + notional_cop * df_end_cop

# El pricer niega estos (convención interna)
usd_nex_pv = -usd_nex_pv_raw
cop_nex_pv = -cop_nex_pv_raw

print('=== Notional Exchange PV ===')
print(f'USD notional exchange PV: {usd_nex_pv:,.2f} USD')
print(f'COP notional exchange PV: {cop_nex_pv:,.2f} COP')

---
## 6. NPV Final y Conversión FX

```
usd_total = usd_leg_pv + usd_notional_exchange_pv
cop_total = cop_leg_pv + cop_notional_exchange_pv

sign = +1 si pay_usd, -1 si receive_usd
NPV_COP = sign * (-usd_total * fx_spot + cop_total)
NPV_USD = NPV_COP / fx_spot
```

In [ ]:
usd_leg_pv = usd_leg_df['pv'].sum()
cop_leg_pv = cop_leg_df['pv'].sum()

usd_total = usd_leg_pv + usd_nex_pv
cop_total = cop_leg_pv + cop_nex_pv

spot = cm.fx_spot
sign = 1.0  # pay_usd = True

npv_cop = sign * (-usd_total * spot + cop_total)
npv_usd = npv_cop / spot

print('=== Componentes del NPV ===')
print(f'USD leg PV:              {usd_leg_pv:>20,.2f} USD')
print(f'USD notional exch PV:    {usd_nex_pv:>20,.2f} USD')
print(f'USD total:               {usd_total:>20,.2f} USD')
print()
print(f'COP leg PV:              {cop_leg_pv:>20,.2f} COP')
print(f'COP notional exch PV:    {cop_nex_pv:>20,.2f} COP')
print(f'COP total:               {cop_total:>20,.2f} COP')
print()
print(f'FX spot:                 {spot:>20,.2f}')
print(f'-USD_total * spot:       {-usd_total * spot:>20,.2f} COP')
print()
print(f'NPV (COP):               {npv_cop:>20,.2f}')
print(f'NPV (USD):               {npv_usd:>20,.2f}')

---
## 7. Comparar con `xccy.price()` (debe dar igual)

In [ ]:
result = xccy.price(**params)
print('=== Output de xccy.price() ===')
for k, v in result.items():
    if isinstance(v, float):
        print(f'  {k:30s}: {v:>20,.4f}')
    else:
        print(f'  {k:30s}: {v}')

print()
print('=== Verificación ===')
print(f'NPV manual vs pricer (COP): {npv_cop:,.2f} vs {result["npv_cop"]:,.2f}')
print(f'Diferencia: {abs(npv_cop - result["npv_cop"]):,.2f}')

---
## 8. Par Xccy Basis

Encontrar el spread (bps) que hace NPV = 0.

In [ ]:
par_basis = xccy.par_xccy_basis(
    notional_usd=10_000_000,
    start_date=datetime(2025, 3, 5),
    maturity_date=datetime(2030, 3, 5),
    fx_initial=4150.0,
)
print(f'Par XCCY basis: {par_basis:.2f} bps')

# Verificar: repricear con par basis debería dar NPV ~ 0
check = xccy.price(
    notional_usd=10_000_000,
    start_date=datetime(2025, 3, 5),
    maturity_date=datetime(2030, 3, 5),
    xccy_basis_bps=par_basis,
    fx_initial=4150.0,
    pay_usd=True,
)
print(f'NPV con par basis: {check["npv_cop"]:,.2f} COP (debe ser ~0)')

---
## 9. Mid-Life Pricing

Para swaps donde `start_date < evaluation_date` (ya están corriendo),
el pricer:
- Construye el schedule completo desde `start_date`
- Filtra períodos pasados
- Clip la primera fecha futura al `curve_floor`
- No incluye el notional exchange inicial (ya se liquidó)

In [ ]:
# Swap que empezó hace 1 año
midlife_result = xccy.price(
    notional_usd=10_000_000,
    start_date=datetime(2024, 3, 5),   # empezó hace ~1 año
    maturity_date=datetime(2029, 3, 5),
    xccy_basis_bps=0.0,
    pay_usd=True,
    fx_initial=3900.0,                  # FX al momento de inception
    usd_spread_bps=-22.0,
)

print('=== Mid-Life Swap (start 2024-03-05, mat 2029-03-05) ===')
print(f'FX inception: {midlife_result["fx_initial"]:,.2f}')
print(f'FX spot:      {midlife_result["fx_spot"]:,.2f}')
print(f'NPV (COP):    {midlife_result["npv_cop"]:>20,.2f}')
print(f'NPV (USD):    {midlife_result["npv_usd"]:>20,.2f}')
print(f'USD leg PV:   {midlife_result["usd_leg_pv"]:>20,.4f}')
print(f'COP leg PV:   {midlife_result["cop_leg_pv"]:>20,.4f}')
print(f'USD nex PV:   {midlife_result["usd_notional_exchange_pv"]:>20,.4f}')
print(f'COP nex PV:   {midlife_result["cop_notional_exchange_pv"]:>20,.4f}')

---
## 10. P&L Decomposition (FX vs Rates)

In [ ]:
pnl = xccy.pnl_inception(
    notional_usd=10_000_000,
    start_date=datetime(2024, 3, 5),
    maturity_date=datetime(2029, 3, 5),
    xccy_basis_bps=0.0,
    pay_usd=True,
    fx_initial=3900.0,
    usd_spread_bps=-22.0,
)

print('=== P&L Decomposition ===')
print(f'NPV total (COP):     {pnl["npv_cop"]:>20,.2f}')
print(f'P&L FX (COP):        {pnl["pnl_fx_cop"]:>20,.2f}')
print(f'P&L Rates (COP):     {pnl["pnl_rates_cop"]:>20,.2f}')
print()
print(f'FX inception:         {pnl["fx_initial"]:>20,.2f}')
print(f'FX spot:              {pnl["fx_spot"]:>20,.2f}')
print(f'FX move:              {pnl["fx_spot"] - pnl["fx_initial"]:>+20,.2f}')

---
## 11. Sensibilidades (DV01 IBR y SOFR)

In [ ]:
# Base NPV
base = xccy.price(
    notional_usd=10_000_000,
    start_date=datetime(2025, 3, 5),
    maturity_date=datetime(2030, 3, 5),
    xccy_basis_bps=0.0,
    pay_usd=True,
    fx_initial=4150.0,
    usd_spread_bps=-22.0,
)
base_npv = base['npv_cop']

# Bump IBR +1bp
cm.bump_ibr(1)
ibr_bumped = xccy.price(
    notional_usd=10_000_000,
    start_date=datetime(2025, 3, 5),
    maturity_date=datetime(2030, 3, 5),
    xccy_basis_bps=0.0,
    pay_usd=True,
    fx_initial=4150.0,
    usd_spread_bps=-22.0,
)
dv01_ibr = ibr_bumped['npv_cop'] - base_npv
cm.bump_ibr(-1)  # reset

# Bump SOFR +1bp
cm.bump_sofr(1)
sofr_bumped = xccy.price(
    notional_usd=10_000_000,
    start_date=datetime(2025, 3, 5),
    maturity_date=datetime(2030, 3, 5),
    xccy_basis_bps=0.0,
    pay_usd=True,
    fx_initial=4150.0,
    usd_spread_bps=-22.0,
)
dv01_sofr = sofr_bumped['npv_cop'] - base_npv
cm.bump_sofr(-1)  # reset

print('=== DV01 (sensibilidad a +1bp) ===')
print(f'Base NPV (COP):   {base_npv:>20,.2f}')
print(f'DV01 IBR (COP):   {dv01_ibr:>20,.2f}')
print(f'DV01 SOFR (COP):  {dv01_sofr:>20,.2f}')
print(f'DV01 Total (COP): {dv01_ibr + dv01_sofr:>20,.2f}')

---
## 12. Comparar Bullet vs Amortizable

In [ ]:
common_params = dict(
    notional_usd=10_000_000,
    start_date=datetime(2025, 3, 5),
    maturity_date=datetime(2030, 3, 5),
    xccy_basis_bps=0.0,
    pay_usd=True,
    fx_initial=4150.0,
    usd_spread_bps=-22.0,
)

bullet_result = xccy.price(**common_params, amortization_type='bullet')
linear_result = xccy.price(**common_params, amortization_type='linear')

comparison = pd.DataFrame([
    {
        'type': 'Bullet',
        'npv_cop': bullet_result['npv_cop'],
        'npv_usd': bullet_result['npv_usd'],
        'usd_leg_pv': bullet_result['usd_leg_pv'],
        'cop_leg_pv': bullet_result['cop_leg_pv'],
        'usd_nex_pv': bullet_result['usd_notional_exchange_pv'],
        'cop_nex_pv': bullet_result['cop_notional_exchange_pv'],
    },
    {
        'type': 'Linear',
        'npv_cop': linear_result['npv_cop'],
        'npv_usd': linear_result['npv_usd'],
        'usd_leg_pv': linear_result['usd_leg_pv'],
        'cop_leg_pv': linear_result['cop_leg_pv'],
        'usd_nex_pv': linear_result['usd_notional_exchange_pv'],
        'cop_nex_pv': linear_result['cop_notional_exchange_pv'],
    },
]).set_index('type')

print('=== Bullet vs Linear Amortization ===')
comparison

In [ ]:
# Par basis para cada tipo
par_bullet = xccy.par_xccy_basis(**common_params, amortization_type='bullet')
par_linear = xccy.par_xccy_basis(**common_params, amortization_type='linear')

print(f'Par basis (Bullet): {par_bullet:+.2f} bps')
print(f'Par basis (Linear): {par_linear:+.2f} bps')
print(f'Diferencia:         {par_linear - par_bullet:+.2f} bps')

---
## 13. Caso Real: GOLOSINAS TRULULU SA — CCS 2026-02-17

Análisis educativo completo del swap confirmado en el PDF CCS 2026.02.16.

### Términos del trade

| Campo | Valor |
|---|---|
| Contraparte A | GOLOSINAS TRULULU SA |
| Contraparte B | Bancolombia |
| Tipo | Cross-Currency Swap (CCS) amortizable |
| Nocional USD | USD 1,000,000 |
| Nocional COP | COP 3,661,000,000 |
| FX Inception | 3,661.00 |
| Inicio | 2026-02-17 |
| Vencimiento | 2028-02-17 |
| Frecuencia | Trimestral |
| Amortización | 8 pagos de 12.5% cada trimestre |
| Pata A (Golosinas **paga**) | SOFR 3M − 22 bps, ACT/360 |
| Pata B (Bancolombia **paga**) | IBR trimestral flat, ACT/360 |

**Perspectiva:** Golosinas **paga USD** (SOFR − 22 bps) y **recibe COP** (IBR). `pay_usd = True`.

### La convención T+2 del SOFR

La curva SOFR se construye con `settlement_days=2`. Esto significa:
- Si `valuation_date = 2026-02-17`, la curva referencia comienza el **2026-02-19** (T+2).
- El primer período del swap arranca el **2026-02-17** → está "antes" de la referencia.
- QuantLib lanzaría `RuntimeError: negative time (-0.00555...)`.

**Fix:** Usar `effective_start = max(periodo_start, sofr_ref_date)` para la consulta de forward rate.
Esto es correcto económicamente: la fijación SOFR del primer cupón empieza en T+2 de la fecha de inception.

In [ ]:
# ── Helper: build CurveManager for a specific historical date ──
def build_cm_for_date(loader, fecha_str: str) -> CurveManager:
    """
    Build a CurveManager for a specific date using the MarketDataLoader.

    market_marks.ibr  → flat dict {'ibr_1d': 9.636, ...}
    build_ibr_curve() → expects    {'ibr_1d': [9.636], ...}  (lists)
    market_marks.sofr → zero rates (can't bootstrap from these)
    sofr_swap_curve   → par coupon rates → use this for SOFR bootstrap
    """
    # IBR: loader.fetch_ibr_quotes() already returns {'ibr_1d': [rate], ...} format
    ibr_db_info = loader.fetch_ibr_quotes(target_date=fecha_str)
    if not ibr_db_info:
        raise ValueError(f'No IBR data for {fecha_str}')
    print(f'  IBR nodes: {list(ibr_db_info.keys())}')

    # SOFR: par swap rates from sofr_swap_curve (NOT market_marks.sofr which has zero rates)
    sofr_df = loader.fetch_sofr_curve(target_date=fecha_str)
    if sofr_df.empty:
        raise ValueError(f'No SOFR data for {fecha_str}')
    print(f'  SOFR par tenors: {list(sofr_df["tenor_months"])}')

    # FX spot: from SET-ICAP intraday data for that date
    fx = loader.fetch_usdcop_spot(target_date=fecha_str)
    print(f'  FX spot: {fx}')

    # Build valuation date
    from datetime import datetime as dt
    d = dt.strptime(fecha_str, '%Y-%m-%d')
    val_date = ql.Date(d.day, d.month, d.year)

    # Build CurveManager
    cm_hist = CurveManager(valuation_date=val_date)
    cm_hist.build_ibr_curve(ibr_db_info)
    cm_hist.build_sofr_curve(sofr_df)
    cm_hist.set_fx_spot(fx)
    return cm_hist


# ── Trade constants (from CCS 2026.02.16 PDF confirmation) ──
T_USD_NOTIONAL  = 1_000_000
T_FX_INCEPTION  = 3_661.00
T_COP_NOTIONAL  = T_USD_NOTIONAL * T_FX_INCEPTION
T_SOFR_SPREAD   = -0.0022           # -22 bps
T_AMORT_FACTOR  = 0.125             # 12.5% each quarter
T_N_PERIODS     = 8
T_START         = ql.Date(17, 2, 2026)
T_MATURITY      = ql.Date(17, 2, 2028)
DC              = ql.Actual360()

# Amortization per period: notional at START of each period
# Period 1: 100%, Period 2: 87.5%, ..., Period 8: 12.5%
T_AMORT_NOTIONALS_USD = [T_USD_NOTIONAL * (1.0 - T_AMORT_FACTOR * i) for i in range(T_N_PERIODS)]
T_AMORT_NOTIONALS_COP = [n * T_FX_INCEPTION for n in T_AMORT_NOTIONALS_USD]

print('=== GOLOSINAS TRULULU SA — Trade Terms ===')
print(f'USD notional:    {T_USD_NOTIONAL:>15,.0f}')
print(f'FX inception:    {T_FX_INCEPTION:>15,.2f}')
print(f'COP notional:    {T_COP_NOTIONAL:>15,.0f}')
print(f'SOFR spread:     {T_SOFR_SPREAD*10000:>15.0f} bps')
print(f'Start:           {T_START}')
print(f'Maturity:        {T_MATURITY}')
print()
print('Amortization schedule:')
for i, (usd, cop) in enumerate(zip(T_AMORT_NOTIONALS_USD, T_AMORT_NOTIONALS_COP)):
    print(f'  Period {i+1}: {(1.0 - T_AMORT_FACTOR*i)*100:5.1f}% → USD {usd:>12,.0f}  |  COP {cop:>18,.0f}')

### 13.1 Construcción del schedule de pagos

Los 8 períodos trimestrales del CCS amortizable:

In [ ]:
# Generate schedule using joint USD+COP calendar
from utilities.colombia_calendar import calendar_colombia
cal_colombia = calendar_colombia()
cal_us = ql.UnitedStates(ql.UnitedStates.FederalReserve)
cal_joint = ql.JointCalendar(cal_colombia, cal_us)

trulu_schedule = ql.Schedule(
    T_START, T_MATURITY,
    ql.Period(3, ql.Months),
    cal_joint,
    ql.Following, ql.Following,
    ql.DateGeneration.Forward, False,
)
trulu_dates = list(trulu_schedule)

print(f'Schedule dates: {len(trulu_dates)}  →  {len(trulu_dates)-1} periods')
print()

period_rows = []
for i in range(1, len(trulu_dates)):
    start = trulu_dates[i-1]
    end   = trulu_dates[i]
    days  = DC.dayCount(start, end)
    tau   = DC.yearFraction(start, end)
    period_rows.append({
        'P': i,
        'start':         str(start),
        'end':           str(end),
        'days':          days,
        'tau (ACT/360)': round(tau, 6),
        'notional_USD':  T_AMORT_NOTIONALS_USD[i-1],
        'notional_COP':  T_AMORT_NOTIONALS_COP[i-1],
    })

pd.DataFrame(period_rows)

### 13.2 Valoración a inception (2026-02-17) con marcas de ese día

**Lo que entra a QuantLib:**
- IBR curve bootstrapped desde `banrep_series_value_v2` (depósitos 1D-12M) + `ibr_2y/5y/10y` (OIS swaps)
- SOFR curve bootstrapped desde `sofr_swap_curve` (par OIS rates 1M-50Y de Eris Futures)
- FX spot = 3,661.00 (del PDF, confirmado por `currency_hour` SET-ICAP)

**Fix del T+2:** El primer período arranca el 17-feb pero la curva SOFR referencia el 19-feb (T+2).
Usamos `effective_start = max(period_start, sofr_curve.referenceDate())` para la query de forward rate.
La `tau` sigue calculándose con las fechas reales del período (17-feb → 17-may = 89 días).

In [ ]:
# ── Build curves for inception date ──
print('--- Loading inception marks (2026-02-17) ---')
cm_inc = build_cm_for_date(loader, '2026-02-17')
print()
print(f'IBR curve reference date:  {cm_inc.ibr_curve.referenceDate()}')
print(f'SOFR curve reference date: {cm_inc.sofr_curve.referenceDate()}')
print()

# Show key discount factors and zero rates
for label, handle in [('IBR', cm_inc.ibr_handle), ('SOFR', cm_inc.sofr_handle)]:
    print(f'{label} discount factors & zero rates (continuous ACT/360):')
    for d in [ql.Date(17,5,2026), ql.Date(17,8,2026), ql.Date(17,2,2027), ql.Date(17,2,2028)]:
        df_ = handle.discount(d)
        zr  = handle.zeroRate(d, DC, ql.Continuous).rate()
        print(f'  {str(d):15s}  DF={df_:.6f}   zero={zr*100:.4f}%')
    print()

In [ ]:
def price_trululu_leg_by_leg(cm_val, val_date: ql.Date, is_inception: bool = True):
    """
    Period-by-period pricing of the Golosinas Trululu CCS.

    Returns (usd_df, cop_df, npv_cop, npv_usd, detail_dict)

    SOFR T+2 fix:
        For any period start before sofr_curve.referenceDate(),
        clip start to referenceDate() for forwardRate() call only.
        tau still uses the actual period dates (economic day count).

    is_inception=True:  include initial notional exchange PV (T_START)
    is_inception=False: initial exchange already settled; only future notional returns
    """
    sofr_ref = cm_val.sofr_curve.referenceDate()
    ibr_ref  = cm_val.ibr_curve.referenceDate()
    spot     = cm_val.fx_spot
    usd_rows, cop_rows = [], []

    for i in range(1, len(trulu_dates)):
        start = trulu_dates[i-1]
        end   = trulu_dates[i]

        if end <= val_date:          # fully elapsed → skip
            continue

        tau   = DC.yearFraction(start, end)
        n_usd = T_AMORT_NOTIONALS_USD[i-1]
        n_cop = T_AMORT_NOTIONALS_COP[i-1]

        # ── USD leg: SOFR − 22 bps ───────────────────────────────────────
        sofr_start = sofr_ref if start < sofr_ref else start
        note = f'T+2 clip: {str(start)}→{str(sofr_ref)}' if sofr_start != start else ''
        fwd_sofr  = cm_val.sofr_handle.forwardRate(sofr_start, end, DC, ql.Simple).rate()
        allin_usd = fwd_sofr + T_SOFR_SPREAD
        cf_usd    = n_usd * allin_usd * tau
        df_usd    = cm_val.sofr_handle.discount(end)
        pv_usd    = cf_usd * df_usd
        usd_rows.append({
            'P': i, 'start': str(start), 'end': str(end),
            'N_USD': n_usd,
            'SOFR_fwd%': round(fwd_sofr * 100, 4),
            'spread_bps': T_SOFR_SPREAD * 10_000,
            'allin%': round(allin_usd * 100, 4),
            'tau': round(tau, 6),
            'CF_USD': round(cf_usd, 2),
            'DF': round(df_usd, 6),
            'PV_USD': round(pv_usd, 2),
            'note': note,
        })

        # ── COP leg: IBR flat ────────────────────────────────────────────
        ibr_start = ibr_ref if start < ibr_ref else start
        fwd_ibr   = cm_val.ibr_handle.forwardRate(ibr_start, end, DC, ql.Simple).rate()
        cf_cop    = n_cop * fwd_ibr * tau
        df_cop    = cm_val.ibr_handle.discount(end)
        pv_cop    = cf_cop * df_cop
        cop_rows.append({
            'P': i, 'start': str(start), 'end': str(end),
            'N_COP': n_cop,
            'IBR_fwd%': round(fwd_ibr * 100, 4),
            'tau': round(tau, 6),
            'CF_COP': round(cf_cop),
            'DF': round(df_cop, 6),
            'PV_COP': round(pv_cop),
        })

    usd_df = pd.DataFrame(usd_rows)
    cop_df = pd.DataFrame(cop_rows)
    usd_leg_pv = usd_df['PV_USD'].sum()
    cop_leg_pv = cop_df['PV_COP'].sum()

    # ── Notional exchange PV ─────────────────────────────────────────────
    # Each quarter: 12.5% of original notional is returned at period end
    usd_nex_pv = 0.0
    cop_nex_pv = 0.0
    for i in range(1, len(trulu_dates)):
        end = trulu_dates[i]
        if end <= val_date:
            continue
        ret_usd = T_USD_NOTIONAL * T_AMORT_FACTOR
        ret_cop = T_COP_NOTIONAL * T_AMORT_FACTOR
        usd_nex_pv += ret_usd * cm_val.sofr_handle.discount(end)
        cop_nex_pv += ret_cop * cm_val.ibr_handle.discount(end)

    if is_inception:
        # At inception: Golosinas pays USD notional, receives COP notional
        # DF(T_START) ≈ 1.0 (today = start)
        df_s_usd = cm_val.sofr_handle.discount(T_START)
        df_s_cop = cm_val.ibr_handle.discount(T_START)
        usd_nex_pv -= T_USD_NOTIONAL * df_s_usd  # pay
        cop_nex_pv += T_COP_NOTIONAL * df_s_cop  # receive

    usd_total = usd_leg_pv + usd_nex_pv
    cop_total = cop_leg_pv + cop_nex_pv
    # Golosinas: receives COP, pays USD
    npv_cop = cop_total - usd_total * spot
    npv_usd = npv_cop / spot
    return usd_df, cop_df, npv_cop, npv_usd, {
        'usd_leg_pv': usd_leg_pv, 'cop_leg_pv': cop_leg_pv,
        'usd_nex_pv': usd_nex_pv, 'cop_nex_pv': cop_nex_pv,
        'usd_total': usd_total, 'cop_total': cop_total,
        'spot': spot, 'npv_cop': npv_cop, 'npv_usd': npv_usd,
    }


print('price_trululu_leg_by_leg() defined.')

In [ ]:
# ── Inception pricing: 2026-02-17 ──────────────────────────────────────────
usd_df_inc, cop_df_inc, npv_cop_inc, npv_usd_inc, det_inc = price_trululu_leg_by_leg(
    cm_inc, val_date=T_START, is_inception=True
)

print('=' * 72)
print('PATA USD — GOLOSINAS PAGA (SOFR 3M − 22 bps)  [inception 2026-02-17]')
print('=' * 72)
print(usd_df_inc.to_string(index=False))
print()
print(f'  USD leg PV:        {det_inc["usd_leg_pv"]:>12,.2f} USD')
print(f'  Notional exch PV:  {det_inc["usd_nex_pv"]:>12,.2f} USD')
print(f'  USD total:         {det_inc["usd_total"]:>12,.2f} USD')
print()
print('=' * 72)
print('PATA COP — BANCOLOMBIA PAGA (IBR trimestral flat)  [inception 2026-02-17]')
print('=' * 72)
print(cop_df_inc.to_string(index=False))
print()
print(f'  COP leg PV:        {det_inc["cop_leg_pv"]:>18,.0f} COP')
print(f'  Notional exch PV:  {det_inc["cop_nex_pv"]:>18,.0f} COP')
print(f'  COP total:         {det_inc["cop_total"]:>18,.0f} COP')

In [ ]:
print('=' * 72)
print('NPV INCEPTION — Perspectiva GOLOSINAS TRULULU SA')
print('=' * 72)
print()
print('  Golosinas PAGA USD (SOFR − 22bps) y RECIBE COP (IBR)')
print()
print('  NPV_COP = COP_total − USD_total × FX_spot')
print(f'          = {det_inc["cop_total"]:,.0f}')
print(f'          − {det_inc["usd_total"]:,.2f} × {det_inc["spot"]:.2f}')
print(f'          = {det_inc["cop_total"] - det_inc["usd_total"] * det_inc["spot"]:,.0f}')
print()
print(f'  NPV (COP): {npv_cop_inc:>18,.0f}')
print(f'  NPV (USD): {npv_usd_inc:>18,.2f}')
print()
print('  Note: at-market swap should be NPV ~ 0 at inception.')
print(f'  Residual = {npv_cop_inc:,.0f} COP reflects bid-offer or curve interpolation.')

### 13.3 Valoración con marcas de hoy (2026-03-03)

El swap lleva 14 días corriendo. Cambios clave:
- **FX spot**: 3,661 → ~3,803 (COP se depreció ~4%)
- **Curvas**: IBR y SOFR pueden haberse movido
- **Notional exchange inicial**: ya se liquidó el 17-feb → `is_inception=False`
- **Cupón período 1 (Feb17→May17)**: parcialmente fijado, pero usamos forward rate del residuo

In [ ]:
# ── Build curves for today ──
print('--- Loading today marks (2026-03-03) ---')
cm_today = build_cm_for_date(loader, '2026-03-03')
print()
print(f'FX move: {T_FX_INCEPTION:.2f} → {cm_today.fx_spot:.2f}  ({(cm_today.fx_spot/T_FX_INCEPTION - 1)*100:+.2f}%)')
print()

# Curve comparison
check_dates = [ql.Date(17,5,2026), ql.Date(17,8,2026), ql.Date(17,2,2027), ql.Date(17,2,2028)]
for label, h_inc, h_tod in [
    ('IBR', cm_inc.ibr_handle, cm_today.ibr_handle),
    ('SOFR', cm_inc.sofr_handle, cm_today.sofr_handle),
]:
    print(f'{label} zero rate shift (continuous, ACT/360):')
    for d in check_dates:
        zr_i = h_inc.zeroRate(d, DC, ql.Continuous).rate()
        zr_t = h_tod.zeroRate(d, DC, ql.Continuous).rate()
        print(f'  {str(d):15s}  {zr_i*100:.4f}% → {zr_t*100:.4f}%  ({(zr_t-zr_i)*10000:+.1f}bps)')
    print()

In [ ]:
# ── Today's mid-life pricing ──
VAL_TODAY = ql.Date(3, 3, 2026)
usd_df_tod, cop_df_tod, npv_cop_tod, npv_usd_tod, det_tod = price_trululu_leg_by_leg(
    cm_today, val_date=VAL_TODAY, is_inception=False
)

print('=' * 72)
print('PATA USD — hoy 2026-03-03  (solo períodos futuros)')
print('=' * 72)
print(usd_df_tod.to_string(index=False))
print()
print(f'  USD leg PV:        {det_tod["usd_leg_pv"]:>12,.2f} USD')
print(f'  Notional exch PV:  {det_tod["usd_nex_pv"]:>12,.2f} USD  (retornos futuros)')
print(f'  USD total:         {det_tod["usd_total"]:>12,.2f} USD')
print()
print('=' * 72)
print('PATA COP — hoy 2026-03-03  (solo períodos futuros)')
print('=' * 72)
print(cop_df_tod.to_string(index=False))
print()
print(f'  COP leg PV:        {det_tod["cop_leg_pv"]:>18,.0f} COP')
print(f'  Notional exch PV:  {det_tod["cop_nex_pv"]:>18,.0f} COP  (retornos futuros)')
print(f'  COP total:         {det_tod["cop_total"]:>18,.0f} COP')

In [ ]:
print('=' * 72)
print('RESUMEN NPV — GOLOSINAS TRULULU SA')
print('=' * 72)
print()
W = 20
print(f'  {"":26s}  {"INCEPTION 17-feb":>{W}}  {"HOY 03-mar":>{W}}  {"DELTA":>14}')
print(f'  {"-"*76}')
print(f'  {"FX spot":26s}  {T_FX_INCEPTION:>{W},.2f}  {cm_today.fx_spot:>{W},.2f}  {cm_today.fx_spot - T_FX_INCEPTION:>+14,.2f}')
print(f'  {"-"*76}')
print(f'  {"USD leg PV":26s}  {det_inc["usd_leg_pv"]:>{W},.2f}  {det_tod["usd_leg_pv"]:>{W},.2f}')
print(f'  {"COP leg PV":26s}  {det_inc["cop_leg_pv"]:>{W},.0f}  {det_tod["cop_leg_pv"]:>{W},.0f}')
print(f'  {"USD notional exch":26s}  {det_inc["usd_nex_pv"]:>{W},.2f}  {det_tod["usd_nex_pv"]:>{W},.2f}')
print(f'  {"COP notional exch":26s}  {det_inc["cop_nex_pv"]:>{W},.0f}  {det_tod["cop_nex_pv"]:>{W},.0f}')
print(f'  {"-"*76}')
print(f'  {"NPV (COP)":26s}  {npv_cop_inc:>{W},.0f}  {npv_cop_tod:>{W},.0f}  {npv_cop_tod - npv_cop_inc:>+14,.0f}')
print(f'  {"NPV (USD)":26s}  {npv_usd_inc:>{W},.0f}  {npv_usd_tod:>{W},.0f}  {npv_usd_tod - npv_usd_inc:>+14,.0f}')
print()
print('  P&L Drivers (14 días):')
print(f'  FX: COP depreció {cm_today.fx_spot - T_FX_INCEPTION:+.0f} COP/USD')
print('    Golosinas paga USD / recibe COP.')
print('    COP más débil = los COP que recibirá tienen menos valor real.')
print('    Pero el notional exchange futuro (recibirá USD, pagará COP a 3,661)')
print('    vale más con COP débil → efecto neto favorable al payer USD.')
print()
print('  Tasas:')
print('    IBR sube  → COP leg (recibe) vale más  → NPV sube')
print('    SOFR baja → USD leg (paga)  vale menos → NPV sube')

In [ ]:
# ── Cross-check against XccySwapPricer (mid-life) ──
# The pricer's price() handles mid-life automatically when start_date < eval_date
xccy_trulu = XccySwapPricer(cm_today)
result_pricer = xccy_trulu.price(
    notional_usd=T_USD_NOTIONAL,
    start_date=datetime(2026, 2, 17),
    maturity_date=datetime(2028, 2, 17),
    xccy_basis_bps=0.0,
    pay_usd=True,
    fx_initial=T_FX_INCEPTION,
    usd_spread_bps=T_SOFR_SPREAD * 10_000,
    cop_spread_bps=0.0,
    amortization_type='custom',
    amortization_schedule=[1.0 - T_AMORT_FACTOR * i for i in range(T_N_PERIODS)],
)
print('=== Cross-check: XccySwapPricer.price() today ===')
print(f'  NPV (COP):    {result_pricer["npv_cop"]:>18,.0f}')
print(f'  NPV (USD):    {result_pricer["npv_usd"]:>18,.2f}')
print()
print(f'  Manual calc:  {npv_cop_tod:>18,.0f}')
print(f'  Diff:         {npv_cop_tod - result_pricer["npv_cop"]:>+18,.0f}')